# Species Name Extraction

Load CSV data, create regex patterns to find species names, then set up blacklists and filters to remove false positives.

In [1]:
import pandas as pd
import re

# Load evaluation data
eval_df = pd.read_csv('evaluationData.csv', encoding='utf-8-sig')
print(f"Loaded {len(eval_df)} records")

# Regex patterns for species identification
patterns = {
    'parentheses': re.compile(r'\(([A-Z][a-z]{2,}\s+[a-z]{3,}(?:\s+[a-z]{3,})?)\)'),
    'binomial': re.compile(r'\b([A-Z][a-z]{2,})\s+([a-z]{3,})\b'),
    'genus_sp': re.compile(r'\b([A-Z][a-z]{2,})\s+sp\.?\b'),
    'author': re.compile(r'\b([A-Z][a-z]{2,})\s+([a-z]{3,})\s+[A-Z][a-z]*')
}

# Words to exclude from species identification
blacklist_words = {
    'Lien', 'vers', 'Ces', 'traits', 'Tome', 'premier', 'Dans', 'Pour', 'Avec', 'Cette', 'Tous', 'Sur', 'Par',
    'Une', 'Des', 'Les', 'The', 'And', 'For', 'With', 'From', 'This', 'That', 'These', 'Those',
    'Argo', 'profilers', 'Details', 'are', 'Morphological', 'variability', 'Antarctic', 'plant',
    'Marine', 'Ocean', 'Sea', 'River', 'Lake', 'Forest', 'Park', 'Coast', 'Bay', 'Data',
    'Ifremer', 'utilisant', 'Image', 'Mesure', 'Distribution', 'Relative', 'Donnees', 'Inventaire',
    'Suivi', 'Etude', 'Analyse', 'Recherche', 'Projet', 'Programme', 'Campagne',
    'France', 'Europe', 'Atlantic', 'Mediterranean', 'Pacific', 'Indian', 'North', 'South',
    'Microwear', 'textures', 'Enhanced', 'Observation', 'Period', 'Global', 'Change', 'Mission',
    'Remote', 'Sensing', 'Systems', 'Advanced', 'Microwave', 'Scanning', 'Radiometer'
}

# Latin word endings for validation
latin_endings = [
    'us', 'a', 'um', 'is', 'e', 'ensis', 'ense', 'icus', 'ica', 'icum',
    'alis', 'ale', 'anus', 'ana', 'anum', 'inus', 'ina', 'inum',
    'osus', 'osa', 'osum', 'eus', 'ea', 'eum', 'arius', 'aria', 'arium',
    'atus', 'ata', 'atum', 'oides', 'formis', 'forme', 'ella', 'ensis',
    'oides', 'ensis', 'ense', 'cola', 'phila', 'philus', 'phaga', 'phagus'
]

# Context patterns to exclude
negative_contexts = [
    r'lien vers', r'données', r'campagne', r'étude', r'projet',
    r'lors de', r'dans le', r'pour le', r'avec le', r'sur le', r'par le'
]

negative_pattern = re.compile('|'.join(negative_contexts), re.IGNORECASE)

Loaded 45 records


In [ ]:

'''is_valid_species() - Checks if a word pair is actually a species name
(length, blacklist, Latin endings, capitalization rules) '''

'''Genus: The first part of a scientific name (capitalized, like "Tursiops")
Epithet: The second part of a scientific name (lowercase, like "truncatus")'''

def is_valid_species(genus, epithet, context=""):
    """Validate genus-epithet pair"""
    if len(genus) < 3 or len(epithet) < 3:
        return False
    
    if genus in blacklist_words or epithet in blacklist_words:
        return False
    
    if context and negative_pattern.search(context):
        return False
    
    has_latin_ending = any(epithet.endswith(end) for end in latin_endings)
    has_vowels = any(v in epithet for v in 'aeiou')
    reasonable_length = 4 <= len(epithet) <= 20
    
    if not (genus[0].isupper() and genus[1:].islower()):
        return False
    
    if not epithet.islower():
        return False
    
    return has_latin_ending or (reasonable_length and has_vowels)

'''extract_species() - Finds all potential species in text using 4 different
patterns, validates each one, returns unique list '''

def extract_species(text):
    """Extract species names from text"""
    if not text:
        return []
    
    found = set()
    
    # Species in parentheses
    for match in patterns['parentheses'].finditer(text):
        species = match.group(1).strip()
        parts = species.split()
        if len(parts) >= 2:
            genus, epithet = parts[0], parts[1]
            start_pos = max(0, match.start() - 30)
            end_pos = min(len(text), match.end() + 30)
            context = text[start_pos:end_pos]
            
            if is_valid_species(genus, epithet, context):
                found.add(f"{genus} {epithet}")
    
    # Species with author citations
    for match in patterns['author'].finditer(text):
        genus, epithet = match.groups()[:2]
        species = f"{genus} {epithet}"
        if species not in found:
            start_pos = max(0, match.start() - 30)
            end_pos = min(len(text), match.end() + 30)
            context = text[start_pos:end_pos]
            
            if is_valid_species(genus, epithet, context):
                found.add(species)
    
    # Regular binomial patterns
    for match in patterns['binomial'].finditer(text):
        genus, epithet = match.groups()
        species = f"{genus} {epithet}"
        if species not in found:
            start_pos = max(0, match.start() - 40)
            end_pos = min(len(text), match.end() + 40)
            context = text[start_pos:end_pos]
            
            if is_valid_species(genus, epithet, context):
                found.add(species)
    
    # Genus sp. format
    for match in patterns['genus_sp'].finditer(text):
        genus = match.group(1)
        if (genus not in blacklist_words and len(genus) >= 4 and 
            genus[0].isupper() and genus[1:].islower()):
            
            start_pos = max(0, match.start() - 30)
            end_pos = min(len(text), match.end() + 30)
            context = text[start_pos:end_pos]
            
            if not negative_pattern.search(context):
                found.add(f"{genus} sp.")
    
    return sorted(list(found))

def calculate_metrics(extracted, ground_truth):
    """Calculate precision, recall, F1"""
    if not extracted and not ground_truth:
        return 1.0, 1.0, 1.0
    if not extracted or not ground_truth:
        return 0.0, 0.0, 0.0
    
    correct = len(extracted.intersection(ground_truth))
    precision = correct / len(extracted)
    recall = correct / len(ground_truth)
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return precision, recall, f1

In [10]:
# Test on sample records
for i in range(min(10, len(eval_df))):
    row = eval_df.iloc[i]
    extracted = extract_species(str(row['Title']) + ' ' + str(row['Description']))
    
    print(f"Record {i+1}: {row['Title'][:60]}...")
    print(f"Found: {extracted}")
    print(f"Ground truth: {row['Species']}\n")

Record 1: Abondance d'huître plate (Ostrea edulis) observée lors des c...
Found: ['Ostrea edulis']
Ground truth: Ostrea edulis


Record 2: Suivi de l'ichtyofaune de la Réserve Naturelle Cerbère-Banyu...
Found: ['Sciaena umbra']
Ground truth: Epinephelus marginatus, Sciaena umbra, Diplodus cervinus

Record 3: Variabilité des traits et de l'environnement stationnel et é...
Found: ['Lyallia kerguelensis']
Ground truth: Lyallia kerguelensis

Record 4: Abondance de petit tacaud (Trisopterus minutus) observée lor...
Found: ['Trisopterus minutus']
Ground truth: Trisopterus minutus


Record 5: Peuplements benthiques subtidaux de la baie de Douarnenez (A...
Found: ['Peuplements benthiques']
Ground truth: Ophiocomina nigra, Ophiothrix fragilis

Record 6: A gridded sea surface salinity data set for the Pacific Ocea...
Found: []
Ground truth: couldn’t find

Record 7: Annually binned Sea Surface Salinity, Temperature and Densit...
Found: ['Annually binned', 'Density data', 'Density time', 'Long gap

Runs extraction on all 45 records, parses ground truth species from CSV, calculates performance metrics for each record, and stores everything in a results dataframe for analysis.

In [4]:
# Run evaluation on all records
results = []

for idx, row in eval_df.iterrows():
    extracted = set(extract_species(str(row['Title']) + ' ' + str(row['Description'])))
    
    gt_text = str(row['Species'])
    ground_truth = set([s.strip() for s in gt_text.split(',') if s.strip() and len(s.split()) >= 2]) if gt_text not in ['nan', "couldn't find", "Couldn't find", "Pas de nom"] else set()
    
    precision, recall, f1 = calculate_metrics(extracted, ground_truth)
    
    results.append({
        'record_id': idx + 1,
        'title': row['Title'][:50] + '...',
        'extracted': ', '.join(sorted(extracted)) if extracted else 'None',
        'ground_truth': ', '.join(sorted(ground_truth)) if ground_truth else 'None',
        'precision': precision,
        'recall': recall,
        'f1': f1
    })

results_df = pd.DataFrame(results)
print(f"Processed {len(results_df)} records")

Processed 45 records


Calculates overall algorithm performance by counting total species found, total correct matches, and total ground truth species across all records, then computes final precision, recall, and F1-score metrics.

In [ ]:
# Calculate overall metrics
total_extracted = sum(len(r['extracted'].split(', ')) if r['extracted'] != 'None' else 0 for r in results)
total_ground_truth = sum(len(r['ground_truth'].split(', ')) if r['ground_truth'] != 'None' else 0 for r in results)
total_correct = sum(len(set(r['extracted'].split(', ')).intersection(set(r['ground_truth'].split(', ')))) 
                   if r['extracted'] != 'None' and r['ground_truth'] != 'None' else 0 for r in results)

precision = total_correct / total_extracted if total_extracted > 0 else 0.0
recall = total_correct / total_ground_truth if total_ground_truth > 0 else 0.0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print("PERFORMANCE METRICS:")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-Score: {f1:.3f}")
print(f"Average F1: {results_df['f1'].mean():.3f}")

print(f"\nExtraction Summary:")
print(f"Total extractions: {total_extracted}")
print(f"Correct extractions: {total_correct}")
print(f"False positives: {total_extracted - total_correct}")

PERFORMANCE METRICS:
Precision: 0.463
Recall: 0.493
F1-Score: 0.477
Average F1: 0.352

Extraction Summary:
Total extractions: 80
Correct extractions: 37
False positives: 43


In [11]:
# Show top performing records
top_results = results_df.nlargest(10, 'f1')
print("Top 10 records by F1-score:")
for i, row in top_results.iterrows():
    print(f"Record {row['record_id']}: F1={row['f1']:.3f}")
    print(f"  Found: {row['extracted']}")
    print(f"  Truth: {row['ground_truth']}\n")

# Save results
results_df.to_csv('species_extraction_results.csv', index=False)
print("Results saved to species_extraction_results.csv")

Top 10 records by F1-score:
Record 1: F1=1.000
  Found: Ostrea edulis
  Truth: Ostrea edulis

Record 3: F1=1.000
  Found: Lyallia kerguelensis
  Truth: Lyallia kerguelensis

Record 4: F1=1.000
  Found: Trisopterus minutus
  Truth: Trisopterus minutus

Record 13: F1=1.000
  Found: Capreolus capreolus, Cervus elaphus, Ovis gmelini, Rupicapra rupicapra
  Truth: Capreolus capreolus, Cervus elaphus, Ovis gmelini, Rupicapra rupicapra

Record 27: F1=1.000
  Found: Castanea crenata, Castanea sativa
  Truth: Castanea crenata, Castanea sativa

Record 29: F1=1.000
  Found: Psammechinus miliaris
  Truth: Psammechinus miliaris

Record 30: F1=1.000
  Found: Solea solea
  Truth: Solea solea

Record 33: F1=1.000
  Found: Jorunna tomentosa
  Truth: Jorunna tomentosa

Record 36: F1=1.000
  Found: Pyrrhocorax pyrrhocorax
  Truth: Pyrrhocorax pyrrhocorax

Record 38: F1=1.000
  Found: Hippoglossoides platessoides
  Truth: Hippoglossoides platessoides

Results saved to species_extraction_results.csv


In [7]:
# Test function
def test_extraction(title, description=""):
    species = extract_species(title + ' ' + description)
    print(f"Input: {title[:50]}...")
    print(f"Found: {species}")
    return species

# Examples
test_extraction("Study of Tursiops truncatus", "Marine mammals (Tursiops truncatus) research")
test_extraction("Lien vers les données", "Information about marine species")

Input: Study of Tursiops truncatus...
Found: ['Tursiops truncatus']
Input: Lien vers les données...
Found: []


[]